# Airbnb Price Prediction

This notebook processes the raw Airbnb dataset, performs data cleaning, handles missing values, encodes categorical variables, scales features, fits regression models, extracts feature weights, and evaluates model performance.


### 1. Import Libraries

Import required packages for data processing, encoding, scaling, model training, evaluation, and serialization.


In [211]:
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor


### 2. Load Dataset

Load the raw dataset from disk and display the first five records.


In [213]:
df = pd.read_csv(r"C:\Users\SART\Desktop\Practise\csvs\archive\train_regression.csv")
df.head()


,id,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,beds,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,5.387292e+07,366516391,4/9/2018,"Chicago, IL",within an hour,96%,95%,f,Back of the Yards,11,13,"['email', 'phone']",t,t,New City,41.794240,-87.661380,Private room in rental unit,Private room,1,1 shared bath,1.0,$30.00,32,1125,32,32,1125,1125,32.0,1125.0,t,30,60,90,365,18,0,0,6/16/2019,8/12/2022,4.94,5.00,4.89,5.00,5.00,4.28,4.83,f,9,1,8,0,0.33
1,3.926966e+07,50276775,12/23/2014,NaN,within an hour,100%,97%,f,Clearwater Beach,1141,2346,"['email', 'phone']",t,t,Near North Side,41.892749,-87.620711,Room in hotel,Private room,12,3 baths,3.0,"$1,247.00",32,365,32,32,365,365,32.0,365.0,t,25,49,66,135,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,t,58,0,58,0,NaN
2,5.564810e+17,681680651,3/8/2020,NaN,within an hour,100%,100%,f,Lake View East,1,1,['phone'],t,t,Lake View,41.948160,-87.652309,Entire rental unit,Entire home/apt,6,1 bath,3.0,$214.00,2,45,2,2,45,45,2.0,45.0,t,0,0,0,0,14,14,1,7/21/2023,12/3/2023,4.87,5.00,5.00,5.00,5.00,4.93,4.80,f,1,1,0,0,2.78
3,5.239728e+07,94345291,10/22/2015,"Chicago, IL",within an hour,100%,98%,f,Bucktown,55,58,"['email', 'phone', 'work_email']",t,t,Hyde Park,41.801030,-87.597160,Entire rental unit,Entire home/apt,2,1 bath,1.0,$59.00,2,180,2,2,180,180,2.0,180.0,t,23,53,83,355,13,13,3,9/2/2023,12/6/2023,4.08,4.08,4.00,4.38,4.23,4.54,3.85,t,55,55,0,0,3.61
4,7.459490e+17,790998106,4/4/2021,"Chicago, IL",within an hour,100%,100%,t,Near South Side,74,75,"['email', 'phone']",t,t,Near South Side,41.855770,-87.624130,Entire rental unit,Entire home/apt,6,2 baths,3.0,$151.00,2,365,2,2,1125,1125,2.0,1125.0,t,0,0,0,0,64,42,3,5/6/2022,12/6/2023,4.80,4.74,4.75,4.83,4.85,4.74,4.72,t,74,74,0,0,3.24


### 3. Clean Formatted Columns

Clean the target price column by removing currency symbols and commas. Remove percentage signs from response rates and extract numeric values from bathrooms text.


In [214]:
# Clean target price column
df['price'] = df['price'].str.replace(r'[\$,]', '', regex=True).astype(float)

# Clean percentage columns
df['host_response_rate'] = df['host_response_rate'].str.replace('%', '', regex=False)
df['host_acceptance_rate'] = df['host_acceptance_rate'].str.replace('%', '', regex=False)

# Clean host location by keeping the primary city name
df['host_location'] = df['host_location'].str.split(',').str[0].str.strip()

# Clean bathrooms text to numeric format (handling half-baths as 0.5)
df['bathrooms_text'] = (
    df['bathrooms_text']
    .str.lower()
    .str.replace('half', '0.5', regex=False)
    .str.extract(r'(\d+(?:\.\d+)?)')[0]
)
df['bathrooms_text'] = pd.to_numeric(df['bathrooms_text'], errors='coerce')


### 4. Extract Verification Features and Drop IDs

Extract binary indicator flags for common verification types and drop non-predictive identifier columns.


In [215]:
# Create indicator columns for verification types
for item in ['email', 'phone', 'work_email']:
    df[f'verif_{item}'] = df['host_verifications'].str.contains(f"'{item}'", na=False).astype(int)

# Drop identifier columns and raw multi-label column
df.drop(columns=['id', 'host_id', 'host_verifications'], inplace=True, errors='ignore')


### 5. Handle Missing Values

Impute missing numeric values with the column mean and categorical values with the column mode.


In [216]:
# Impute numeric missing values with mean
numeric_cols = df.select_dtypes(include='number').columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

# Impute categorical missing values with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

# Drop any remaining unhandled rows
df.dropna(inplace=True)


C:\Users\SART\AppData\Local\Temp\ipykernel_11592\384377837.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


### 6. Categorical Encoding

Transform categorical text features into numeric values using LabelEncoder.


In [217]:
label_encoders = {}
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

df.head()


C:\Users\SART\AppData\Local\Temp\ipykernel_11592\4265658737.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,host_listings_count,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,beds,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,verif_email,verif_phone,verif_work_email
0,1030,23,3,34,67,0,9,11,13,1,1,49,41.794240,-87.661380,28,2,1,1.0,1.0,30.0,32,1125,32,32,1125,1125,32.0,1125.0,1,30,60,90,365,18,0,0,1136,444,4.940000,5.00000,4.890000,5.000000,5.000000,4.280000,4.830000,0,9,1,8,0,0.330000,1,1,0
1,531,23,3,1,69,0,35,1141,2346,1,1,46,41.892749,-87.620711,36,2,12,3.0,3.0,1247.0,32,365,32,32,365,365,32.0,365.0,1,25,49,66,135,0,0,0,1242,152,4.759538,4.79342,4.747335,4.841084,4.853426,4.745506,4.668381,1,58,0,58,0,1.928421,1,1,0
2,873,23,3,1,1,0,87,1,1,1,1,37,41.948160,-87.652309,10,0,6,1.0,3.0,214.0,2,45,2,2,45,45,2.0,45.0,1,0,0,0,0,14,14,1,1332,201,4.870000,5.00000,5.000000,5.000000,5.000000,4.930000,4.800000,0,1,1,0,0,2.780000,0,1,0
3,229,23,3,1,70,0,22,55,58,1,1,33,41.801030,-87.597160,10,0,2,1.0,1.0,59.0,2,180,2,2,180,180,2.0,180.0,1,23,53,83,355,13,13,3,1674,209,4.080000,4.08000,4.000000,4.380000,4.230000,4.540000,3.850000,1,55,55,0,0,3.610000,1,1,1
4,1004,23,3,1,1,1,111,74,75,1,1,47,41.855770,-87.624130,10,0,6,2.0,3.0,151.0,2,365,2,2,1125,1125,2.0,1125.0,1,0,0,0,0,64,42,3,1073,209,4.800000,4.74000,4.750000,4.830000,4.850000,4.740000,4.720000,1,74,74,0,0,3.240000,1,1,0


### 7. Train-Test Split

Split the dataset into features (X) and target (y), followed by an 80/20 train-test partition.


In [218]:
X = df.drop(columns='price')
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


### 8. Feature Scaling

Fit the StandardScaler on training features and apply transformation to both training and test sets.


In [219]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reconstruct DataFrames to retain feature column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)


### 9. Train Linear Regression and Inspect Feature Weights

Fit a Linear Regression model on the scaled features and inspect the weights assigned to each feature.


In [220]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

# Create DataFrame of feature weights
weights_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Weight': linear_model.coef_
}).sort_values(by='Weight', ascending=False)

print("Top 10 Features Increasing Price (Positive Weights):")
print(weights_df.head(10))

print("\nTop 10 Features Decreasing Price (Negative Weights):")
print(weights_df.tail(10))

print(f"\nModel Intercept: ${linear_model.intercept_:.2f}")


Top 10 Features Increasing Price (Positive Weights):
                           Feature      Weight
45  calculated_host_listings_count  364.345153
7              host_listings_count  147.552140
16                    accommodates   75.117111
6               host_neighbourhood   73.282943
36                     last_review   68.106715
18                            beds   62.007114
12                        latitude   58.235827
29                 availability_60   53.370538
13                       longitude   44.576354
14                   property_type   42.014620

Top 10 Features Decreasing Price (Negative Weights):
                                        Feature      Weight
43                          review_scores_value  -33.558384
20                               maximum_nights  -40.074202
5                             host_is_superhost  -43.590650
11                       neighbourhood_cleansed  -44.102370
33                        number_of_reviews_ltm  -46.786322
15              

### 10. Model Evaluation

Compute regression evaluation metrics including Mean Squared Error, Root Mean Squared Error, Mean Absolute Error, and R-squared.


In [221]:
y_pred = linear_model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE):        {mse:.2f}")
print(f"Root Mean Squared Error (RMSE):  ${rmse:.2f}")
print(f"Mean Absolute Error (MAE):       ${mae:.2f}")
print(f"R-squared Score (R2):            {r2 * 100:.2f}%")


Mean Squared Error (MSE):        41618.15
Root Mean Squared Error (RMSE):  $204.01
Mean Absolute Error (MAE):       $146.20
R-squared Score (R2):            -37.55%


### 11. Tree-Based Models and Feature Importances

Train a DecisionTreeRegressor and a RandomForestRegressor to compare non-linear model performance and evaluate feature importances.


In [222]:
# Decision Tree Regressor
dt_model = DecisionTreeRegressor(max_depth=10, min_samples_leaf=20, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
print(f"Decision Tree RMSE: ${np.sqrt(mean_squared_error(y_test, dt_pred)):.2f}")

# Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
print(f"Random Forest RMSE: ${np.sqrt(mean_squared_error(y_test, rf_pred)):.2f}")

# Top 10 most important features from Random Forest
rf_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print("\nTop 10 Feature Importances (Random Forest):")
print(rf_importance.head(10))


Decision Tree RMSE: $264.54
Random Forest RMSE: $855.64

Top 10 Feature Importances (Random Forest):
                      Feature  Importance
36                last_review    0.185051
24     maximum_maximum_nights    0.162977
17             bathrooms_text    0.139561
6          host_neighbourhood    0.066125
20             maximum_nights    0.061984
16               accommodates    0.056421
10     host_identity_verified    0.045780
23     minimum_maximum_nights    0.030294
39  review_scores_cleanliness    0.024497
31           availability_365    0.023764


### 12. Save Models and Scaler

Serialize the trained models and standard scaler to disk using pickle for future deployment.


In [223]:
with open('linear_model.pkl', 'wb') as f:
    pickle.dump(linear_model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("Models and scaler saved successfully.")


Models and scaler saved successfully.
